# Post-convergence route conversion on real language data

Jointly-trained recurrent–attention hybrid LM (GRU backbone + causal retrieval branch, additive logits, shared tied output projection), trained from scratch on TinyStories.

Pipeline (mirrors the paper's 30M protocol, on language):

1. **Free hybrid** — train with λ=0 until converged. Check retrieval is load-bearing (deletion must hurt).
2. **Viability screen** — train a recurrent-only model with the same backbone. Record ratio to hybrid *before* steering.
3. **Late price** — from the free checkpoint, price retrieval with several λ (never arm continues at λ=0).
4. **Deletion test** — remove retrieval from never vs late arms; report perplexity and in-context copy accuracy.

Metrics logged at every eval: `ppl`, `copy_acc` (induction/copy accuracy: positions where the correct next token is the continuation seen earlier in the context), `share_c` (retrieval contribution share), `D_h`/`D_c` (resampling dependence on copy_acc), `del_cost` (copy_acc drop when retrieval deleted), `del_ppl`.

**How to run:** set `SMOKE=True` first (≈5 min on any GPU) to verify the pipeline. Then `SMOKE=False` on an A100 (≈3–5 days total for defaults). Everything is written to `runs_langhybrid/`; at the end a `results.zip` is produced — send me that.

If the go/no-go cell after Phase A says retrieval is not load-bearing or recurrence is not viable, change `ctx` (shorter → recurrence more viable, attention less load-bearing; longer → opposite) and rerun Phase A. Don't run Phase B until both checks pass.

In [2]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
SMOKE = False   # <-- set False for the real run

CFG = dict(
    dataset="roneneldan/TinyStories", n_stories=400_000, tokenizer="gpt2",
    ctx=256,                      # start short; recurrence must be viable here
    d=512, n_trunk=4, d_ff=2048,  # position-wise MLP trunk, then one GRU layer (as in the 30M model)
    n_heads=8, d_c=512,           # retrieval branch
    batch=32, lr=3e-4, warmup=500, clip=1.0,
    free_steps=20_000,            # phase A (free hybrid and recurrent-only screen)
    price_steps=10_000,           # phase B (never / late arms, constant LR)
    seeds=[0, 1, 2],
    lambdas=[0.03, 0.1, 0.3],     # price is lambda * mean_t ||l_c,t||_2 / sqrt(V); sweep, then keep the one that crosses at high acc
    eval_every=1000, eval_batches=25,
    out_dir="runs_langhybrid",
)
if SMOKE:
    CFG.update(n_stories=5_000, free_steps=300, price_steps=200, seeds=[0], lambdas=[0.1],
               eval_every=100, eval_batches=3, batch=16)
CFG

{'dataset': 'roneneldan/TinyStories',
 'n_stories': 400000,
 'tokenizer': 'gpt2',
 'ctx': 256,
 'd': 512,
 'n_trunk': 4,
 'd_ff': 2048,
 'n_heads': 8,
 'd_c': 512,
 'batch': 32,
 'lr': 0.0003,
 'warmup': 500,
 'clip': 1.0,
 'free_steps': 20000,
 'price_steps': 10000,
 'seeds': [0, 1, 2],
 'lambdas': [0.03, 0.1, 0.3],
 'eval_every': 1000,
 'eval_batches': 25,
 'out_dir': 'runs_langhybrid'}

In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118


In [4]:
# !pip install -q datasets transformers matplotlib
import os, json, math, time, random, zipfile, glob
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer

device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
os.makedirs(CFG["out_dir"], exist_ok=True)
LOGDIR = os.path.join(CFG["out_dir"], "logs"); os.makedirs(LOGDIR, exist_ok=True)
CKDIR  = os.path.join(CFG["out_dir"], "ckpt"); os.makedirs(CKDIR, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(torch.__version__, torch.cuda.get_device_name(0))

2.10.0+cu128 Tesla T4


## Data

In [5]:
tok = AutoTokenizer.from_pretrained(CFG["tokenizer"])
V = len(tok)
cache = os.path.join(CFG["out_dir"], f'tokens_{CFG["n_stories"]}.npy')
if os.path.exists(cache):
    tokens = np.load(cache)
else:
    ds = load_dataset(CFG["dataset"], split="train").select(range(CFG["n_stories"]))
    def enc(b):
        return {"ids": [t + [tok.eos_token_id] for t in tok(b["text"])["input_ids"]]}
    ds = ds.map(enc, batched=True, remove_columns=ds.column_names, num_proc=min(8, os.cpu_count()))
    tokens = np.concatenate([np.asarray(x, dtype=np.uint16) for x in ds["ids"]])
    np.save(cache, tokens)
n_val = int(0.02 * len(tokens))
train_tok, val_tok = tokens[:-n_val], tokens[-n_val:]
print(f"vocab {V}, train tokens {len(train_tok)/1e6:.1f}M, val tokens {len(val_tok)/1e6:.2f}M")

def get_batch(arr, B, T, rng):
    ix = rng.integers(0, len(arr) - T - 1, size=B)
    x = torch.from_numpy(np.stack([arr[i:i+T+1] for i in ix]).astype(np.int64)).to(device)
    return x[:, :-1], x[:, 1:]

def induction_mask(x, y):
    # position t is a "copy" position if x_t appeared earlier at s<t and x_{s+1} == y_t (most recent occurrence)
    xn, yn = x.cpu().numpy(), y.cpu().numpy()
    m = np.zeros_like(xn, dtype=bool)
    for b in range(xn.shape[0]):
        seen = {}
        for t in range(xn.shape[1]):
            cur = xn[b, t]
            if cur in seen and seen[cur] == yn[b, t]:
                m[b, t] = True
            seen[cur] = yn[b, t]
    return torch.from_numpy(m).to(device)

_vrng = np.random.default_rng(1234)
VAL = [get_batch(val_tok, CFG["batch"], CFG["ctx"], _vrng) for _ in range(CFG["eval_batches"])]
VAL_MASK = [induction_mask(x, y) for x, y in VAL]
print("copy positions per batch:", np.mean([m.sum().item() for m in VAL_MASK]))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/400000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1058 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1039 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1097 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1106 > 1024). Running this sequence through the model will result in indexing errors


vocab 50257, train tokens 88.1M, val tokens 1.80M
copy positions per batch: 726.4


## Model — identifiable dual-route readout for LM

In [8]:
class Block(nn.Module):
    def __init__(s, d, dff):
        super().__init__(); s.ln = nn.LayerNorm(d); s.fc1 = nn.Linear(d, dff); s.fc2 = nn.Linear(dff, d)
    def forward(s, x): return x + s.fc2(F.gelu(s.fc1(s.ln(x))))

class Hybrid(nn.Module):
    """logits_t = W f(h_t) + W g(c_t);  c_t = Attn(q=h_t, K/V = h_{<t}) (strictly causal, learned sink for t=0).
    W is the tied output projection shared by both routes; l_h and l_c stay additively separable."""
    def __init__(s, V, d, n_trunk, dff, n_heads, d_c, retrieval=True):
        super().__init__()
        s.emb = nn.Embedding(V, d)
        nn.init.normal_(s.emb.weight, std=0.02)
        s.trunk = nn.ModuleList([Block(d, dff) for _ in range(n_trunk)])
        s.ln_in = nn.LayerNorm(d)
        s.gru = nn.GRU(d, d, batch_first=True)
        s.f = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU())
        s.retrieval = retrieval
        if retrieval:
            s.nh, s.d_c = n_heads, d_c
            s.q = nn.Linear(d, d_c, bias=False); s.k = nn.Linear(d, d_c, bias=False); s.v = nn.Linear(d, d_c, bias=False)
            s.sink_k = nn.Parameter(torch.zeros(1, 1, d_c)); s.sink_v = nn.Parameter(torch.zeros(1, 1, d_c))
            s.o = nn.Linear(d_c, d_c, bias=False)
            s.g = nn.Sequential(nn.LayerNorm(d_c), nn.Linear(d_c, d), nn.GELU())
        s.out = nn.Linear(d, V, bias=False); s.out.weight = s.emb.weight

    def states(s, x):
        h = s.emb(x)
        for b in s.trunk: h = b(h)
        with torch.autocast("cuda", enabled=False):
            h, _ = s.gru(s.ln_in(h).float())
        return h

    def context(s, h):
        B, T, _ = h.shape
        q = s.q(h).view(B, T, s.nh, -1).transpose(1, 2)
        k = torch.cat([s.sink_k.expand(B, 1, -1).to(h.dtype), s.k(h)], 1).view(B, T+1, s.nh, -1).transpose(1, 2)
        v = torch.cat([s.sink_v.expand(B, 1, -1).to(h.dtype), s.v(h)], 1).view(B, T+1, s.nh, -1).transpose(1, 2)
        mask = torch.ones(T, T+1, dtype=torch.bool, device=h.device).tril(0)  # col0 = sink, col j = state j-1 -> strictly < t
        att = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return s.o(att.transpose(1, 2).reshape(B, T, s.d_c))

    def forward(s, x, perm_h=None, perm_c=None, beta=1.0, retrieval_on=True):
        h = s.states(x)
        c = s.context(h) if (s.retrieval and retrieval_on) else None   # clean context computed before any permutation
        lh = s.out(s.f(h if perm_h is None else h[perm_h]))
        if c is None: return lh, None
        lc = beta * s.out(s.g(c if perm_c is None else c[perm_c]))
        return lh, lc

def make(retrieval=True):
    m = Hybrid(V, CFG["d"], CFG["n_trunk"], CFG["d_ff"], CFG["n_heads"], CFG["d_c"], retrieval).to(device)
    n = sum(p.numel() for p in m.parameters())
    print(f"{'hybrid' if retrieval else 'recurrent-only'}: {n/1e6:.2f}M params")
    return m

def loss_fn(lh, lc, y, lam):
    logits = lh if lc is None else lh + lc
    ce = F.cross_entropy(logits.reshape(-1, V).float(), y.reshape(-1))
    price = torch.zeros((), device=y.device) if lc is None else lc.float().norm(dim=-1).mean() / math.sqrt(V)
    return ce + lam * price, ce, price

## Metrics: perplexity, copy accuracy, resampling dependence, contribution share, deletion

In [12]:
@torch.no_grad()
def evaluate(model, perm=None, retrieval_on=True, beta=1.0):
    model.eval()
    g = torch.Generator(device=device).manual_seed(0)
    tot, n, correct, cnt, sh, sc = 0.0, 0, 0, 0, [], []
    for (x, y), m in zip(VAL, VAL_MASK):
        B = x.shape[0]
        ph = torch.randperm(B, generator=g, device=device) if perm == "h" else None
        pc = torch.randperm(B, generator=g, device=device) if perm == "c" else None
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lh, lc = model(x, perm_h=ph, perm_c=pc, beta=beta, retrieval_on=retrieval_on)
        logits = (lh if lc is None else lh + lc).float()
        tot += F.cross_entropy(logits.reshape(-1, V), y.reshape(-1), reduction="sum").item(); n += y.numel()
        pred = logits.argmax(-1)
        correct += (pred[m] == y[m]).sum().item(); cnt += m.sum().item()
        if lc is not None and perm is None:
            sh.append(lh.float().std(-1).mean().item()); sc.append(lc.float().std(-1).mean().item())
    model.train()
    out = dict(loss=tot / n, ppl=math.exp(tot / n), copy_acc=correct / max(cnt, 1))
    if sh: out["share_c"] = float(np.mean(sc) / (np.mean(sh) + np.mean(sc)))
    return out

def route_report(model):
    clean = evaluate(model)
    rep = dict(ppl=clean["ppl"], loss=clean["loss"], copy_acc=clean["copy_acc"], share_c=clean.get("share_c", float("nan")))
    a = clean["copy_acc"]
    D = lambda ar: float(np.clip((a - ar) / max(a, 1e-6), 0, 1))
    rh = evaluate(model, perm="h")
    rep.update(D_h=D(rh["copy_acc"]), dloss_h=rh["loss"] - clean["loss"])
    if model.retrieval:
        rc = evaluate(model, perm="c")
        de = evaluate(model, retrieval_on=False)
        rep.update(D_c=D(rc["copy_acc"]), dloss_c=rc["loss"] - clean["loss"],
                   del_ppl=de["ppl"], del_copy=de["copy_acc"], del_cost=a - de["copy_acc"], del_dloss=de["loss"] - clean["loss"])
    return rep

## Training loop (warmup then constant LR, so late arms are comparable to never)

In [14]:
def train(model, steps, lam, seed, tag, start_step=0, opt=None):
    if opt is None:
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], betas=(0.9, 0.95), weight_decay=0.0)
    rng = np.random.default_rng(seed * 100_003 + start_step)
    logf = open(os.path.join(LOGDIR, f"{tag}.jsonl"), "a")
    t0 = time.time(); model.train()
    for step in range(start_step, start_step + steps):
        for gp in opt.param_groups: gp["lr"] = CFG["lr"] * min(1.0, (step + 1) / CFG["warmup"])
        x, y = get_batch(train_tok, CFG["batch"], CFG["ctx"], rng)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lh, lc = model(x)
        loss, ce, price = loss_fn(lh, lc, y, lam)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"]); opt.step()
        last = step == start_step + steps - 1
        if step % CFG["eval_every"] == 0 or last:
            rep = route_report(model)
            rep.update(step=step, tag=tag, lam=lam, train_ce=ce.item(), price=price.item(), sec=time.time() - t0)
            logf.write(json.dumps(rep) + "\n"); logf.flush()
            print(f"[{tag}] step {step:6d} ce {ce.item():.3f} price {price.item():.3f} | ppl {rep['ppl']:.2f} copy {rep['copy_acc']:.3f} "
                  f"share_c {rep.get('share_c', float('nan')):.3f} D_h {rep['D_h']:.2f} D_c {rep.get('D_c', float('nan')):.2f} del_cost {rep.get('del_cost', float('nan')):.3f}")
    logf.close()
    return opt

def save(model, opt, name): torch.save({"model": model.state_dict(), "opt": opt.state_dict()}, os.path.join(CKDIR, name))
def load(model, name):
    ck = torch.load(os.path.join(CKDIR, name), map_location=device)
    model.load_state_dict(ck["model"])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], betas=(0.9, 0.95), weight_decay=0.0); opt.load_state_dict(ck["opt"])
    return opt

def last_log(tag):
    with open(os.path.join(LOGDIR, f"{tag}.jsonl")) as f: lines = f.readlines()
    return json.loads(lines[-1])

## Phase A — free hybrid + recurrent-only viability screen

In [ ]:
for seed in CFG["seeds"]:
    if not os.path.exists(os.path.join(CKDIR, f"free_s{seed}.pt")):
        set_seed(seed); hyb = make(True)
        opt = train(hyb, CFG["free_steps"], 0.0, seed, f"free_s{seed}")
        save(hyb, opt, f"free_s{seed}.pt"); del hyb, opt; torch.cuda.empty_cache()
    if not os.path.exists(os.path.join(CKDIR, f"reconly_s{seed}.pt")):
        set_seed(seed); rec = make(False)
        opt = train(rec, CFG["free_steps"], 0.0, seed, f"reconly_s{seed}")
        save(rec, opt, f"reconly_s{seed}.pt"); del rec, opt; torch.cuda.empty_cache()

hybrid: 37.29M params
[free_s0] step      0 ce 10.858 price 0.161 | ppl 51842.21 copy 0.000 share_c 0.518 D_h 0.00 D_c 0.00 del_cost -0.000
[free_s0] step   1000 ce 2.784 price 5.974 | ppl 15.85 copy 0.666 share_c 0.298 D_h 0.89 D_c 0.20 del_cost 0.085
[free_s0] step   2000 ce 2.448 price 8.314 | ppl 11.16 copy 0.742 share_c 0.334 D_h 0.87 D_c 0.30 del_cost 0.154
[free_s0] step   3000 ce 2.375 price 8.780 | ppl 9.71 copy 0.771 share_c 0.308 D_h 0.87 D_c 0.30 del_cost 0.176
[free_s0] step   4000 ce 2.187 price 10.423 | ppl 8.91 copy 0.787 share_c 0.319 D_h 0.87 D_c 0.32 del_cost 0.186
[free_s0] step   5000 ce 2.008 price 11.964 | ppl 8.45 copy 0.792 share_c 0.316 D_h 0.86 D_c 0.33 del_cost 0.195
[free_s0] step   6000 ce 2.065 price 12.021 | ppl 8.12 copy 0.799 share_c 0.321 D_h 0.85 D_c 0.34 del_cost 0.197
[free_s0] step   7000 ce 1.886 price 15.322 | ppl 7.88 copy 0.807 share_c 0.339 D_h 0.84 D_c 0.35 del_cost 0.203
[free_s0] step   8000 ce 1.961 price 14.151 | ppl 7.70 copy 0.808 shar

## Go / no-go before Phase B

In [ ]:
rows = []
for seed in CFG["seeds"]:
    h, r = last_log(f"free_s{seed}"), last_log(f"reconly_s{seed}")
    rows.append(dict(seed=seed, hyb_ppl=h["ppl"], hyb_copy=h["copy_acc"], share_c=h["share_c"], D_h=h["D_h"], D_c=h["D_c"],
                     del_cost=h["del_cost"], del_ppl=h["del_ppl"],
                     rec_ppl=r["ppl"], rec_copy=r["copy_acc"],
                     viab_copy=r["copy_acc"] / max(h["copy_acc"], 1e-6), viab_ppl=h["ppl"] / r["ppl"]))
import pandas as pd
df = pd.DataFrame(rows); print(df.round(3).to_string(index=False))
lb = df.del_cost.mean() > 0.10 and df.D_c.mean() > 0.5
vi = df.viab_copy.mean() >= 0.95 and df.viab_ppl.mean() >= 0.95
print()
print("LOAD-BEARING incumbent (deletion hurts, D_c high):", "YES" if lb else "NO  -> lengthen ctx or make task more retrieval-heavy")
print("VIABLE alternative (recurrent-only >= 0.95 of hybrid on copy_acc and ppl):", "YES" if vi else "NO  -> shorten ctx")
print("PROCEED TO PHASE B:", "YES" if (lb and vi) else "NO")

## Phase B — never vs late price, from the same free checkpoint and optimizer state

In [ ]:
arms = [("never", 0.0)] + [(f"late_l{lam}", lam) for lam in CFG["lambdas"]]
for seed in CFG["seeds"]:
    for name, lam in arms:
        tag = f"{name}_s{seed}"
        if os.path.exists(os.path.join(CKDIR, f"{tag}.pt")): continue
        set_seed(seed); m = make(True); opt = load(m, f"free_s{seed}.pt")
        opt = train(m, CFG["price_steps"], lam, seed, tag, start_step=CFG["free_steps"], opt=opt)
        save(m, opt, f"{tag}.pt"); del m, opt; torch.cuda.empty_cache()

## Deletion test + summary (this is the table that matters)

In [ ]:
rows = []
for seed in CFG["seeds"]:
    for name, lam in arms:
        e = last_log(f"{name}_s{seed}")
        rows.append(dict(arm=name, lam=lam, seed=seed, ppl=e["ppl"], copy_acc=e["copy_acc"], share_c=e["share_c"],
                         D_h=e["D_h"], D_c=e["D_c"], inverted=e["D_h"] > e["D_c"],
                         del_cost=e["del_cost"], del_ppl=e["del_ppl"], del_ppl_ratio=e["del_ppl"] / e["ppl"]))
res = pd.DataFrame(rows)
agg = res.groupby("arm").agg(lam=("lam","first"), ppl=("ppl","mean"), copy_acc=("copy_acc","mean"), share_c=("share_c","mean"),
                             D_h=("D_h","mean"), D_c=("D_c","mean"), inverted=("inverted","sum"), n=("seed","count"),
                             del_cost=("del_cost","mean"), del_cost_sd=("del_cost","std"), del_ppl_ratio=("del_ppl_ratio","mean"))
print(agg.round(3).to_string())
never = agg.loc["never"]
print("\nSUCCESS CRITERION per late arm: copy_acc >= 0.95*never, inverted in all seeds, del_cost < 0.01, del_ppl_ratio < 1.05")
for arm in agg.index:
    if arm == "never": continue
    a = agg.loc[arm]
    ok = (a.copy_acc >= 0.95 * never.copy_acc) and (a.inverted == a.n) and (a.del_cost < 0.01) and (a.del_ppl_ratio < 1.05)
    print(f"  {arm}: {'PASS' if ok else 'fail'}   (never arm del_cost={never.del_cost:.3f}, del_ppl_ratio={never.del_ppl_ratio:.2f})")
res.to_csv(os.path.join(CFG["out_dir"], "final_table.csv"), index=False)
json.dump(dict(cfg=CFG, phaseA=df.to_dict("records"), phaseB=res.to_dict("records"), agg=agg.reset_index().to_dict("records")),
          open(os.path.join(CFG["out_dir"], "summary.json"), "w"), indent=1)

## Plots

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(arms), figsize=(4 * len(arms), 3.2), sharey=True)
for i, (name, lam) in enumerate(arms):
    for seed in CFG["seeds"]:
        L = [json.loads(l) for l in open(os.path.join(LOGDIR, f"{name}_s{seed}.jsonl"))]
        s = [r["step"] for r in L]
        ax[i].plot(s, [r["D_h"] for r in L], "-", color="C0", alpha=.7, label="D_h" if seed == CFG["seeds"][0] else None)
        ax[i].plot(s, [r["D_c"] for r in L], "-", color="C1", alpha=.7, label="D_c" if seed == CFG["seeds"][0] else None)
        ax[i].plot(s, [r["copy_acc"] for r in L], "--", color="C2", alpha=.7, label="copy_acc" if seed == CFG["seeds"][0] else None)
    ax[i].set_title(f"{name}"); ax[i].set_xlabel("step")
ax[0].legend(); plt.tight_layout(); plt.savefig(os.path.join(CFG["out_dir"], "trajectories.png"), dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(agg.index, agg.del_cost, yerr=agg.del_cost_sd.fillna(0)); ax.set_ylabel("copy_acc drop when retrieval deleted")
plt.xticks(rotation=30); plt.tight_layout(); plt.savefig(os.path.join(CFG["out_dir"], "deletion.png"), dpi=130); plt.show()

## Package results — send me `results.zip`

In [ ]:
zp = os.path.join(CFG["out_dir"], "results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(LOGDIR, "*.jsonl")): z.write(f, os.path.join("logs", os.path.basename(f)))
    for f in ["summary.json", "final_table.csv", "trajectories.png", "deletion.png"]:
        p = os.path.join(CFG["out_dir"], f)
        if os.path.exists(p): z.write(p, f)
print("wrote", zp, f"{os.path.getsize(zp)/1e3:.0f} KB")